# State injection

An injection cube (`"I"`, or `LeafCubeKind.INJECTION`) starts a logical column in one
of eight single-qubit states, prepared by a small encoder on the patch. The
encoder is not fault tolerant: a single fault in it corrupts the injected state.

In [ ]:
from tqec import NoiseModel, compile_block_graph
from tqec.computation.block_graph import BlockGraph
from tqec.computation.cube import LeafCubeKind
from tqec.utils.injection_state import INJECTION_STATES
from tqec.utils.position import Position3D

print(list(INJECTION_STATES))

## An injection column

An injection cube must sit directly below a regular cube, joined by a temporal
pipe. Its observable is random by design: the correlation surface ends open at the
injection cube.

In [ ]:
def injection_column(state: str) -> BlockGraph:
    g = BlockGraph(f"injection of {state}")
    g.add_cube(Position3D(0, 0, 0), "I", state=state)
    g.add_cube(Position3D(0, 0, 1), "XZX")
    g.add_pipe(Position3D(0, 0, 0), Position3D(0, 0, 1))
    return g


g = injection_column("i")
g.view_as_html(show_correlation_surface=g.find_correlation_surfaces()[0])

In [ ]:
compiled = compile_block_graph(g)
compiled.generate_crumble_url(k=1, add_polygons=True)

## Magic states

`T` and `T_DAG` are not stim gates, so `generate_stim_circuit` (and Crumble) refuse
them. `generate_stim_text` emits the circuit as text, with the real gate, for a
simulator that supports it.

In [ ]:
text = compile_block_graph(injection_column("T")).generate_stim_text(k=1)
print([line for line in text.splitlines() if line.split()[:1] == ["T"]])

## Noise

Since the encoder is not fault tolerant, it dominates the logical error rate.
`noiseless_injection=True` leaves the encoder's own qubits noiseless during its
moments, to measure the error rate of everything downstream.

In [ ]:
noise = NoiseModel.uniform_depolarizing(1e-3)
for noiseless in (False, True):
    circuit = compiled.generate_stim_circuit(k=1, noise_model=noise, noiseless_injection=noiseless)
    n_noise = sum(1 for inst in circuit.flattened() if inst.name.startswith("DEPOLARIZE"))
    print(f"noiseless_injection={noiseless}: {n_noise} depolarizing instructions")

## Beside other cubes

An injection cube may share its z-slice with cubes that keep running: here it
starts a column that merges into a running one.

In [ ]:
column = [Position3D(0, 0, z) for z in range(3)]
injection, merged = Position3D(1, 0, 0), Position3D(1, 0, 1)

g_beside = BlockGraph("injection beside a column")
for position in column:
    g_beside.add_cube(position, "XZX")
g_beside.add_cube(injection, "I")
g_beside.add_cube(merged, "XZX")
for u, v in [(column[0], column[1]), (column[1], column[2]), (injection, merged), (column[1], merged)]:
    g_beside.add_pipe(u, v)

g_beside.view_as_html()

In [ ]:
compile_block_graph(g_beside).generate_crumble_url(k=1, add_polygons=True)

It may also share a slice with a Y-basis measurement (a `Y_HALF_CUBE` cap) on
another column.

In [ ]:
memory, y_cap = Position3D(0, 0, 0), Position3D(0, 0, 1)
injection, above = Position3D(1, 0, 0), Position3D(1, 0, 1)

g_y = BlockGraph("injection beside a Y cap")
g_y.add_cube(memory, "XZX")
g_y.add_cube(y_cap, LeafCubeKind.Y_HALF_CUBE)
g_y.add_cube(injection, "I")
g_y.add_cube(above, "XZX")
g_y.add_pipe(memory, y_cap)
g_y.add_pipe(injection, above)

compile_block_graph(g_y).generate_crumble_url(k=1, add_polygons=True)